# Modelling: first model to predict the age of an abalone

We predict `Rings` from the physical measurements, using the conclusions of `eda.ipynb`:
- rename the columns to snake_case,
- remove the 2 rows with `height == 0`,
- one-hot encode `sex` with a `DictVectorizer` (same approach as the course),
- train a `LinearRegression` and evaluate it with the RMSE.

The code is written as small functions so it can be moved to `src/modelling` in the next step.
Experiments are tracked with MLflow (the `mlruns/` folder is not pushed to the repository).

# Imports

In [ ]:
import mlflow
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

# Config

In [ ]:
DATA_PATH = "../data/abalone.csv"
MLFLOW_TRACKING_URI = "../mlruns"
EXPERIMENT_NAME = "abalone-age-prediction"

TARGET = "rings"
CATEGORICAL_COLS = ["sex"]
NUMERICAL_COLS = ["length", "diameter", "height", "whole_weight", "shucked_weight", "viscera_weight", "shell_weight"]
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Preprocessing

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load the abalone dataset from a csv file."""
    return pd.read_csv(path)


def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Rename the columns to snake_case (e.g. 'Whole weight' -> 'whole_weight')."""
    return df.rename(columns=lambda col: col.lower().replace(" ", "_"))


def filter_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """Remove the abalones with a height of 0 (data errors)."""
    return df[df["height"] > 0]


def encode_categorical_cols(df: pd.DataFrame, categorical_cols: list[str] | None = None) -> pd.DataFrame:
    """Cast categorical columns to strings so the DictVectorizer one-hot encodes them."""
    if categorical_cols is None:
        categorical_cols = CATEGORICAL_COLS
    df = df.copy()
    df[categorical_cols] = df[categorical_cols].astype("str")
    return df


def extract_x_y(
    df: pd.DataFrame, dv: DictVectorizer | None = None, with_target: bool = True
) -> tuple[csr_matrix, np.ndarray | None, DictVectorizer]:
    """Turn the dataframe into a feature matrix (and target), fitting the DictVectorizer if none is given."""
    dicts = df[CATEGORICAL_COLS + NUMERICAL_COLS].to_dict(orient="records")
    if dv is None:
        dv = DictVectorizer()
        dv.fit(dicts)
    x = dv.transform(dicts)
    y = df[TARGET].values if with_target else None
    return x, y, dv

In [ ]:
df = load_data(DATA_PATH)
df = rename_columns(df)
df = filter_outliers(df)
df = encode_categorical_cols(df)
print(df.shape)
df.head()

In [ ]:
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)

X_train, y_train, dv = extract_x_y(train_df)
X_test, y_test, _ = extract_x_y(test_df, dv=dv)

print(f"Train: {X_train.shape}, test: {X_test.shape}")
print(f"Features: {dv.feature_names_}")

# Model

In [ ]:
def train_model(x: csr_matrix, y: np.ndarray) -> LinearRegression:
    """Train a linear regression."""
    model = LinearRegression()
    model.fit(x, y)
    return model


def predict(x: csr_matrix, model: LinearRegression) -> np.ndarray:
    """Predict the number of rings."""
    return model.predict(x)


def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute the RMSE."""
    return root_mean_squared_error(y_true, y_pred)

In [ ]:
model = train_model(X_train, y_train)

train_rmse = evaluate_model(y_train, predict(X_train, model))
test_rmse = evaluate_model(y_test, predict(X_test, model))
baseline_rmse = evaluate_model(y_test, np.full(len(y_test), y_train.mean()))

print(f"Train RMSE: {train_rmse:.3f}")
print(f"Test RMSE: {test_rmse:.3f}")
print(f"Baseline RMSE (always predict the mean): {baseline_rmse:.3f}")

The linear regression predicts the number of rings with an error of about 2.2 rings on the test set,
much better than always predicting the mean (≈ 3.4). Train and test RMSE are close, so there is no overfitting.

# Experiment tracking with MLflow

We log each experiment (parameters, metrics and model) to compare a linear regression with a random forest.

To see the runs, from the repository root: `uv run mlflow ui`, then open http://localhost:5000.

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
def run_experiment(model: LinearRegression | RandomForestRegressor, model_name: str) -> float:
    """Train a model, log its parameters, metrics and the model itself to MLflow, and return the test RMSE."""
    with mlflow.start_run(run_name=model_name):
        mlflow.set_tags({"model": model_name, "data": DATA_PATH})
        mlflow.log_params(model.get_params())
        mlflow.log_params({"test_size": TEST_SIZE, "random_state": RANDOM_STATE})

        model.fit(X_train, y_train)
        y_pred = predict(X_test, model)
        rmse = evaluate_model(y_test, y_pred)
        mlflow.log_metrics({"rmse": rmse, "mae": mean_absolute_error(y_test, y_pred)})
        mlflow.sklearn.log_model(model, artifact_path="model")
    return rmse

In [ ]:
experiments = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE),
}

for model_name, candidate in experiments.items():
    rmse = run_experiment(candidate, model_name)
    print(f"{model_name}: test RMSE = {rmse:.3f}")

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME], order_by=["metrics.rmse ASC"])
runs[["tags.mlflow.runName", "metrics.rmse", "metrics.mae", "start_time"]]

# Conclusion

- **Chosen model: `LinearRegression`**, test RMSE ≈ 2.2 rings (MAE ≈ 1.6), against ≈ 3.4 for the mean baseline.
- The random forest gets almost the same RMSE (≈ 2.2) with a much bigger and slower model, so it is not worth it for a first version.
- Preprocessing to reuse in `src/modelling`: `rename_columns` → `filter_outliers` → `encode_categorical_cols` → `extract_x_y` (`DictVectorizer`), then `train_model` / `predict` / `evaluate_model`.
- The fitted `DictVectorizer` and the model are the two objects to save (pickle) for the API.